<a href="https://colab.research.google.com/github/BrenooOliveira/UBO_Indentification/blob/main/demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

NameError: name 'pyspark' is not defined

In [ ]:
!wget https://repos.spark-packages.org/graphframes/graphframes/0.8.4-spark3.5-s_2.12/graphframes-0.8.4-spark3.5-s_2.12.jar

--2025-12-15 22:06:13--  https://repos.spark-packages.org/graphframes/graphframes/0.8.4-spark3.5-s_2.12/graphframes-0.8.4-spark3.5-s_2.12.jar
Resolving repos.spark-packages.org (repos.spark-packages.org)... 3.170.19.31, 3.170.19.20, 3.170.19.75, ...
Connecting to repos.spark-packages.org (repos.spark-packages.org)|3.170.19.31|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 247575 (242K) [binary/octet-stream]
Saving to: ‘graphframes-0.8.4-spark3.5-s_2.12.jar.1’

graphframes-0.8.4-s 100%[===================>] 241.77K  --.-KB/s    in 0.04s   

2025-12-15 22:06:13 (6.40 MB/s) - ‘graphframes-0.8.4-spark3.5-s_2.12.jar.1’ saved [247575/247575]



In [ ]:
!pip uninstall -y pyspark
!pip install pyspark==3.5.1
!pip install graphframes


Found existing installation: pyspark 4.0.1
Uninstalling pyspark-4.0.1:
  Successfully uninstalled pyspark-4.0.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 8.9 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.5.1-py2.py3-none-any.whl size=317488493 sha256=f47a30f2604216e9df25f9d1740e008952fd114b1305b93524efb2573f0a25aa
  Stored in directory: /root/.cache/pip/wheels/b1/91/5f/283b53010a8016a4ff1c4a1edd99bbe73afacb099645b5471b
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.9
    Uninstalling py4j-0.10.9.9:
      Successfully uninstalled py4j-0.10.9.9
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.0.1 requires pyspark[connect]~=4.0.0, b

In [ ]:
from pyspark.sql import SparkSession
from graphframes import GraphFrame

spark = SparkSession.builder \
    .appName("GraphFramesSpark4") \
    .config(
        "spark.jars",
        "/content/graphframes-0.8.4-spark3.5-s_2.12.jar"
    ) \
    .getOrCreate()



In [ ]:
vertices_data = [
    ("CNPJ_001", "PJ", "Empresa Alpha"),
    ("CNPJ_002", "PJ", "Empresa Beta"),
    ("CNPJ_003", "PJ", "Empresa Gamma"),
    ("CPF_001", "PF", "João Silva"),
    ("CPF_002", "PF", "Maria Souza"),
]

vertices_df = spark.createDataFrame(
    vertices_data,
    ["id", "tipo", "nome"]
)

vertices_df.show()


+--------+----+-------------+
|      id|tipo|         nome|
+--------+----+-------------+
|CNPJ_001|  PJ|Empresa Alpha|
|CNPJ_002|  PJ| Empresa Beta|
|CNPJ_003|  PJ|Empresa Gamma|
| CPF_001|  PF|   João Silva|
| CPF_002|  PF|  Maria Souza|
+--------+----+-------------+



In [ ]:
edges_data = [
    # PF → PJ
    ("CPF_001", "CNPJ_001", "SOCIO"),
    ("CPF_002", "CNPJ_002", "SOCIO"),

    # PJ → PJ
    ("CNPJ_002", "CNPJ_001", "SOCIO"),
    ("CNPJ_003", "CNPJ_002", "SOCIO"),
]

edges_df = spark.createDataFrame(
    edges_data,
    ["src", "dst", "relacao"]
)

edges_df.show()


+--------+--------+-------+
|     src|     dst|relacao|
+--------+--------+-------+
| CPF_001|CNPJ_001|  SOCIO|
| CPF_002|CNPJ_002|  SOCIO|
|CNPJ_002|CNPJ_001|  SOCIO|
|CNPJ_003|CNPJ_002|  SOCIO|
+--------+--------+-------+



In [ ]:
g = GraphFrame(vertices_df, edges_df)

/usr/local/lib/python3.12/dist-packages/pyspark/sql/dataframe.py:168: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(


In [ ]:
g.vertices.show()

+--------+----+-------------+
|      id|tipo|         nome|
+--------+----+-------------+
|CNPJ_001|  PJ|Empresa Alpha|
|CNPJ_002|  PJ| Empresa Beta|
|CNPJ_003|  PJ|Empresa Gamma|
| CPF_001|  PF|   João Silva|
| CPF_002|  PF|  Maria Souza|
+--------+----+-------------+



In [ ]:
# 🔍 Encontrar beneficiários finais (PF no topo)
# PF que não é controlada por ninguém (não aparece como dst)

beneficiarios_finais = g.vertices.filter(
    (g.vertices.tipo == "PF") &
    (~g.vertices.id.isin(
        [row.dst for row in g.edges.select("dst").distinct().collect()]
    ))
)

beneficiarios_finais.show()


+-------+----+-----------+
|     id|tipo|       nome|
+-------+----+-----------+
|CPF_001|  PF| João Silva|
|CPF_002|  PF|Maria Souza|
+-------+----+-----------+



In [ ]:
# 🔎 Encontrar LINHAGEM COMPLETA (cadeias PJ → PJ → PF)
# BFS – Busca até encontrar PF
paths = g.bfs(
    fromExpr="id = 'CNPJ_001'",
    toExpr="tipo = 'PF'",
    maxPathLength=5
)

paths.show(truncate=False)


/usr/local/lib/python3.12/dist-packages/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


+---+----+----+
|id |tipo|nome|
+---+----+----+
+---+----+----+



In [ ]:
ciclos = g.find("(a)-[e]->(b); (b)-[f]->(a)")
ciclos.show()


+---+---+---+---+
|  a|  e|  b|  f|
+---+---+---+---+
+---+---+---+---+



### 🌐 3. EXPORTANDO PARA GRAFO INTERATIVO (HTML)

In [ ]:
v_pd = vertices_df.toPandas()
e_pd = edges_df.toPandas()

In [ ]:
import networkx as nx

G = nx.DiGraph()

# Vértices
for _, row in v_pd.iterrows():
    G.add_node(
        row["id"],
        label=row["nome"],
        tipo=row["tipo"]
    )

# Arestas
for _, row in e_pd.iterrows():
    G.add_edge(row["src"], row["dst"], label=row["relacao"])


In [ ]:
#!pip install pyvis
from pyvis.network import Network

net = Network(
    height="750px",
    width="100%",
    directed=True,
    notebook=True
)

for node, data in G.nodes(data=True):
    color = "lightblue" if data["tipo"] == "PJ" else "lightgreen"
    net.add_node(
        node,
        label=data["label"],
        color=color
    )

for src, dst, data in G.edges(data=True):
    net.add_edge(src, dst, label=data["label"])

net.show("grafo_societario.html")


grafo_societario.html
